# CoordAnalyst - an educational tool to analyse coordination complexes


Claire Bachmann, Bertille Delloye, Enéa Drezet--Marçot, Victor Davril


## Introduction

Coordination chemistry is an active field of research, with its study sitting at the intersection of many different branches of science, and its impact being major. In medicine, coordination compounds are indispensable. Cisplatin, one of the most widely used anticancer drugs in the world, is a simple platinum complex discovered in the 1960s that remains a frontline treatment for testicular, ovarian, and lung cancers. Another exmaple is Hemoglobin, the protein that carries oxygen through our blood, an iron coordination complex. In catalysis and industry, coordination chemistry drives some of the most economically significant chemical processes on the planet.The production of ammonia, acetaldehyde or even plastics, using the Haber-Bosch process, the Wacker process, and Ziegler-Natta catalysts, relies on iron, palladium and aluminium catalysts. In materials science, coordination compounds called metal-organic frameworks (MOFs) have emerged as one of the most exciting research areas of the 21st century, with applications in gas storage, carbon capture, drug delivery, and chemical sensing. Their extraordinary surface areas and tunable pore geometries make them uniquely powerful materials. In biology, nearly a third of all known proteins contain a metal ion at their active site : these coordination complexes perform catalysis, electron transfer, or structural stabilisation. Understanding these metalloenzymes is central to drug design and our understanding of life itself.  
Despite this enormous importance, coordination compounds remain challenging to characterise. Infrared and Raman spectroscopy are among the primary experimental tools used to study them, in order to reveal coordination modes, identify ligands, confirm geometries, and monitor reactions in real time. Yet interpreting these spectra requires deep chemical knowledge and access to reference data that is scattered across literature.  
Coordination chemistry is taught as a Bachelor's course in chemistry as the concepts are extremely relevant, and sometimes intertwine with subjects such as spectroscopy and organometallic chemistry. Students are expected to aquire an understanding of metal and ligands, oxidation states, geometry, d-electrons, and bond stretches.    
Our package provides an educational tool with which students can try out different complexes' names and formulas and verify their properties, as well as visualize them in two and three dimensions. Moreover, CoordAnalyst provides an approximate IR and Raman spectra of the complex based on reference data from Nakamoto tables. It is presented as a unified interface, designed to make the spectral characterisation of coordination compounds more accessible to students and academics.



## Material and methods

### 1.1 Software architecture

The project is organised as a Python package, with the main source code stored in `src/coordchem`. The chemical data used by the program is kept outside the main source code, to separate the package functions from the reference data. Inside `coordchem`, the core modules are responsible for interpreting coordination complexes from user input. The file `parser.py` defines `parse_formula()`, which extracts the metal, ligands, charge, counter-ions, oxidation state and coordination number from a formula. The file `name.py` defines `parse_name()`, which performs a similar task starting from a coordination compound name. Both functions return the same structured object, `ParsedComplex`.

The file `complex.py` provides a higher-level interface through the `Complex` class. This class uses the formula and name parsers through methods such as `Complex.from_formula()`, `Complex.from_name()` and `Complex.from_input()`. It gives direct access to the main properties of the complex and connects the parsed information to the geometry, visualisation and spectra modules.

The `geometry.py` module adds the chemical interpretation step. It contains functions such as `get_d_count()`, `get_geometry()`, `predict_geometry()` and `geometry_report()`. These functions use the parsed information to calculate the metal d-electron count and predict an idealised coordination geometry from the coordination number, metal, oxidation state and ligand information.

Two subpackages were created inside `coordchem`: `viz` and `spectra`. The `viz` subpackage contains the tools used for molecular visualisation. For 2D drawings, `ligand_data.py` stores ligand SMILES strings, donor atom indices and display labels. The file `diagram_2d.py` contains the main drawing functions, including `parse_complex_input()`, `build_coordination_mol()`, `diagram_2d_svg()` and `save_diagram_2d()`. The file `layout_2d.py` defines the idealised coordination-site positions with functions such as `coordination_sites()`, `chelate_octahedral_sites()`, `edta_octahedral_sites()` and `mixed_polydentate_site_groups()`. The file `transform_2d.py` contains the mathematical transformations used to position ligands, especially polydentate ligands, with functions such as `transform_monodentate()`, `transform_polydentate()`, `transform_acac()`, `transform_oxalate()` and `transform_edta()`.

The 3D visualisation is handled in `molecule3D.py`. This file defines idealised 3D coordination-site positions for different geometries, such as linear, tetrahedral, square planar and octahedral arrangements. These positions are selected with `geometry_positions()` and used by `build_complex_3d()` to assemble the metal and ligands into an RDKit molecule with a 3D conformer. The resulting structure can then be displayed interactively with `view_complex_3d()` or exported as HTML with `complex_3d_html()`.

The `spectra` subpackage contains the tools used for IR and Raman prediction. The `predictor.py` file extracts relevant spectral bands from the database for an input `ParsedComplex` and returns them as a `PredictionResult` object, ordered by increasing wavenumber. The `renderer.py` file then converts these bands into a simulated spectrum by representing each band as a Gaussian peak and summing the peaks to produce the final plotted spectrum.

### 1.2 Formula and Name Parser, Properties Extraction

The first step of the program is to transform the user input into a structured object that can be reused by the rest of the package. A coordination complex can be written as a formula, such as `[Fe(CN)6]4-`, or as a name, such as `tetraamminecopper(II)`. Although these two inputs look different, both are converted into the same internal format: a `ParsedComplex` object. This object stores the main chemical information of the complex, including the metal centre, ligands, ligand charges, denticity, donor atoms, complex charge, counter-ions, oxidation state and coordination number.

For formulas, this work is handled by `parser.py`, mainly through the function `parse_formula(formula: str) -> ParsedComplex`. This function reads the formula step by step. It separates possible counter-ions from the coordination sphere, identifies the metal centre, detects and counts the ligands, and extracts the global charge of the complex. The parser then enriches the result using internal dictionaries of known ligands, counter-ions and metals. From this information, it calculates derived properties such as the total ligand charge, the oxidation state of the metal and the coordination number.

For names, the same idea is implemented in `name.py` with the function `parse_name(name: str) -> ParsedComplex`. Instead of reading brackets and charge symbols, this function searches the compound name for known ligand names, metal names, prefixes and oxidation states written in Roman numerals. For example, in `tetraamminecopper(II)`, the program recognises `ammine` as the ligand, `copper` as the metal and `(II)` as the oxidation state. The name parser then returns the same `ParsedComplex` format as the formula parser. This parser is not intended to understand every possible IUPAC name, but it works for the common coordination names covered by the ligand and metal tables.

To make the package easier to use, the parsed information is wrapped in a higher-level `Complex` object defined in `complex.py`. This class acts as the main interface of the package. The methods `Complex.from_formula()`, `Complex.from_name()` and `Complex.from_input()` allow the user to create the same type of object from different input formats. The `Complex` object gives direct access to useful properties such as the metal, ligands, oxidation state, coordination number, predicted geometry and d-electron count.

Once the complex has been parsed, the information can be passed to other modules. In particular, `geometry.py` uses functions such as `get_geometry()` and `get_d_count()` to predict the idealised geometry of the complex and calculate the number of d-electrons on the metal. This common structure is important because it allows the geometry, visualisation and spectra modules to work from the same chemical representation instead of handling formulas and names separately.


### 1.3 Spectral Band Database
The data was extracted systematically from  Nakamoto, K. "Infrared and Raman Spectra of Inorganic and Coordination Compounds", 6th ed., Wiley, 2009. It was placed in SEED_BANDS, a list of tuples containing the informations (ligand, coordination, metal, spectrum_type, wn_min, wn_max, intensity, assignment, ir_active, raman_active, source), which is defined at the top of the file as a BandRecord object. The data is grouped by ligand for easy verification, and for both IR and Raman stretches.

Then the class `IRBandDB` is defined :  
The __init__ method does three things in order every time:  
-Opens a connection to SQLite  
-Creates the ir_bands table if it doesn't exist yet (_create_table)  
-Seeds the table with data if it's empty (_seed)  
   
First, _create_table runs a CREATE TABLE IF NOT EXISTS SQL statement. It defines the columns: ligand, coordination mode, metal, spectrum type, wavenumber range, intensity, assignment, whether it's IR/Raman active, and the source reference. Also creates two indexes to make lookups faster.
Then, _seed inserts all the rows from the SEED_BANDS list at the top of the file. It uses executemany which inserts all rows in one operation rather than one by one. The row count is checked first so it never double-inserts.  

The main query method is `get_bands()`.   
(a) builds a SQL query with WHERE ligand = ? and AND spectrum_type = ?. The ? placeholders are essential as they prevent SQL injection and handle special characters safely.  
(b) if a coordination filter like "terminal" is passed, it adds AND coordination = ? to the query _row_to_record which return a list of BandRecord.  
(c) if a specific metal is passed : It splits the results into two groups: rows specific to that metal (e.g. metal = "Fe") and generic rows (metal = "any"). It then keeps all the specific rows, and only adds generic rows whose assignment isn't already covered by a specific row. This is how, for example, the Fe-specific 2093 cm⁻¹ band takes priority over the generic 2100–2200 range for the same C≡N stretch.  
`get_all_ligands()` returns a simple list of all ligand symbols in the database, e.g. ["CN", "CO", "Cl", "NH3", ...], which is useful for checking which complexes' spectra can be splotted.  
`get_bands_in_range(,)` is used for reverse lookup and can be used to identify an unknown peak in an experimental spectrum.  
`add_band()` is for inserting a custom entry. Useful when you encounter a ligand not in the seed data. I takes all the same fields as the database columns and commits immediately.  
`summary()`(ligand) loops over IR then Raman bands and prints them as a formatted table, used for debugging and demos.  

### 1.4 Spectrum Prediction - Gaussians and plotting
**predictor.py** has a class `PredictionResult` which contains the attributes bands, intensities, the type of spectrum, the metal, the complex formula, ligand_coverage and warnings.  
The main function is `predict_spectrum()` which returns a `PredictionResult` object from a `ParsedComplex` object. It loops over every ligand in the complex using the query db.get_bands for the specific ligand, spectrum and metal. Chemical corrections are applied to those bands by calling three different functions :    
- Backbonding shifts for π-accepting ligands (CN, CO, NO) : These ligands accept electron density from the metal via π-backbonding. More electron-rich metals (low oxidation state) allow for more backbonding, the C≡N / C≡O bond is weaker ; the wavenumber of the stretch is lower (negative shift). Similarly, less electron-rich metals (high oxidation state) mean less backbonding, a stronger bond in the ligands and a higher wavenumber for the strectch (positive shift).
- Coordination shifts account for the shift of the wavenumber from the free ligand to the ligand bound to the metal center.  
- Selection rules impose which M-L band types are IR/Raman active depending on their geometry. Based on the mutual exclusion rule: in centrosymmetric complexes (Oh, D4h), a band cannot be both IR and Raman active.

The band intensity is scaled according to the number of ligand of the same type. The bands extracted are sorted by order of increasing wavenumber.     
   
**renderer.py** first defines the mathematical gaussian with `build_spectrum()`for variables x, center, height and sigma. In build_spectrum : first we use linspace for x to create an array of evenly spaced numbers between wn_range[0] and wn_range[1]. We have decided for 3000 points, as the more points we have, the smoother the curve appears. For y, originally we create an array of the same size containing only zero. Then, looping over every band, a gaussian peak is added to the right y position.  
`plot_spectrum()` uses x, y from build_spectrum and traces the figure. A vertical dotted line is also added from the top of the peak to the position on the x-axis to be able to easily read off the wavenumber of the stretch. 

### 1.5 Two-Dimensional Visualisation

The two-dimensional visualisation module generates schematic drawings of coordination complexes. After the complex has been parsed and its geometry has been predicted, the program uses this information to place the metal center at the center of the drawing and arrange the ligands around it according to an idealised coordination geometry. The aim is to provide a clear and chemically meaningful representation of the coordination sphere, which will not necessarily correspond to an exact crystallographic structure.

The 2D drawing workflow is mainly handled in **diagram_2d.py**. The function `parse_complex_input()` allows the drawing functions to accept a formula, a compound name or an already parsed `ParsedComplex` object. The predicted geometry is then obtained with `get_geometry()` and used to decide how the ligands should be arranged around the metal.

The positions of the coordination sites are generated in **layout_2d.py** by the function `coordination_sites(geometry, n_sites)`. Depending on the predicted geometry, this function returns predefined positions around the metal center. Linear complexes are drawn with two opposite sites, square planar complexes as a flat cross, tetrahedral complexes with wedge and dashed bonds to suggest depth, and octahedral complexes with a simplified projection containing two axial bonds and four surrounding bonds. For coordination number 8, the module can also generate a square-antiprismatic-type layout.

The drawing also depends on ligand-specific information stored in **ligand_data.py**. This file contains the SMILES strings used to describe each ligand, donor atom indices, and display labels. A SMILES string is a compact text representation of molecular connectivity. For example, ethylenediamine (`en`) is represented as `NCCN`, oxalate (`ox`) as `[O-]C(=O)C(=O)[O-]`, and acetylacetonate (`acac`) as `CC(=O)C=C(O)C`. These SMILES strings are converted by RDKit into `Chem.Mol` objects, which are connectivity-based molecular representations that RDKit can use to generate 2D coordinates and draw the ligand.

Small monodentate ligands, such as NH3, H2O, CN or Cl, can be drawn as compact labels to keep the diagram readable. Larger or polydentate ligands, such as ethylenediamine, oxalate, acetylacetonate or EDTA, are drawn using their RDKit-generated ligand structures. This allows the program to show the ligand backbone when it is chemically useful, while keeping the overall diagram clear.

The layout is based on simple coordination chemistry rules. The coordination number defines an ideal reference geometry: linear for coordination number 2, trigonal planar for coordination number 3, tetrahedral or square planar for coordination number 4, octahedral for coordination number 6, and square antiprismatic or dodecahedral for coordination number 8. However, real coordination complexes are rarely perfectly ideal. Their structures can depend on metal size, d-electron count, ligand steric effects and electronic interactions. For example, square planar geometries are often favoured for low-spin d⁸ metal centres such as Pt(II), Pd(II) or Ni(II) with strong-field ligands, while octahedral complexes may show distortions such as tetragonal elongation, tetragonal compression or Jahn-Teller distortion.

In our implementation, we chose a simplified idealised model rather than trying to reproduce these distortions quantitatively. The main objective was to avoid overlap between ligands and to keep the diagram easy to interpret. For this reason, specific layout functions were added for cases that are difficult to draw with a generic geometry. For example, `chelate_octahedral_sites()` is used for tris-bidentate octahedral complexes, `tridentate_octahedral_sites()` for bis-tridentate complexes, and `edta_octahedral_sites()` for EDTA-like hexadentate coordination.

A more general function, `mixed_polydentate_site_groups()`, was also added for complexes containing both polydentate and monodentate ligands. In this case, the polydentate ligands are assigned neighbouring coordination sites first, because they need several adjacent positions to bind correctly. The remaining sites are then filled by the monodentate ligands. This improves the readability of mixed-ligand complexes and reduces overlap problems.

After the coordination sites are chosen, the ligand structures are placed onto these sites. This is done in `build_coordination_mol()`, which creates a composite RDKit molecule containing the metal and all ligands. The function `_expand_ligands()` expands the ligand dictionary into a complete ligand list. Then `_make_ligand_mol()` takes the SMILES string from **ligand_data.py** and converts it into an RDKit `Chem.Mol` object with 2D coordinates. The function `_match_donor_indices()` identifies which atom or atoms of the ligand bind to the metal.

The ligand is then positioned using the transformation functions from `transform_2d.py`. Monodentate ligands are placed with `transform_monodentate()`, while polydentate ligands are placed with `transform_polydentate()`. Some ligands required more specific treatment to make the final drawing clearer, so special functions were added: `transform_acac()` for acetylacetonate, `transform_oxalate()` for oxalate, and `transform_edta()` for EDTA. These functions adjust the ligand orientation, scale and position so that the final diagram is readable and significant.

The final drawing is generated by `diagram_2d_svg()`. This function builds the full coordination molecule, draws it with RDKit, adds a title and returns the result as an SVG image. The SVG format was chosen because it gives clean and scalable figures that can easily be included in reports or presentations. The function `save_diagram_2d()` can then be used to save the SVG file.

Bond styles are used to make the drawings easier to interpret. Normal lines represent bonds in the drawing plane, while bold wedges and dashed bonds suggest approximate three-dimensional orientation. This is especially useful for tetrahedral and octahedral drawings. However, this remains a simplified convention. The current 2D module does not explicitly distinguish stereochemical isomers such as cis/trans, mer/fac or Λ/Δ. 

Overall, the 2D visualisation module follows a clear workflow: the input is parsed with `parse_complex_input()`, the geometry is predicted with `get_geometry()`, ligand-specific information is taken from **ligand_data.py**, coordination sites are generated by `coordination_sites()` or special layout functions, ligands are positioned with the transformation functions, and the final SVG drawing is produced by `diagram_2d_svg()` or saved with `save_diagram_2d()`.

### 1.5 Three-Dimentional Visualisation

The 3D visualisation is composed of various parts that allow to obtain a 3D visualisation of the complex given.

First, the geometry of the complex was implemented using the following functions: `octahedral_positions()`, `octahedral_positions_bidentate()`, `octahedral_positions_tridentate()`, `tetrahedral_positions()`, `square_planar_positions()`, `linear_positions()`, `trigonal_planar_positions()`, 
`trigonal_bipyramidal_positions()`. Thus, it works for octahedral, linear, triangular planar, tetrahedral, trigonal_planar, trigonal_bipyramidal complexes, composed of monodentate, bidentate or tridentate ligands.  
The geometry of the complex is found with `geometry_positions()` and the corresponding geometry function is attributed to the complex with the dictionary _GEOMETRY_BUILDERS.

From the name or the formula input, the metal is identified. Since the metal center is only one atom it can directly be added in a RDKit molecule. Nevertheless, the ligands are often polyatomics, meaning that their smile is needed. Then they can be added and placed around the metal according to the complex geometry.
Thus in a second part RDKit ligands are build using the smile of the ligands in the function `build_ligand_3D()`. It adds the H atoms and gives its 3D coordinates.
The donor atom is found by comparing the atoms in the ligand with the symbol of the donor atom from the ligands dictionnary KNOWN_LIGANDS in the function `find_donor_atom()`.
However, additional functions needed to be implemented for ambidentate ligands that have at least two possible donor atoms such as thiocynato and nitro/nitrito: `thiocynato_ligand_positions()`, `nitrito_ligand_positions()`. Since they have more than one donor atom, the formula of the ligand found with the **name.py** functions is used to identify the donor atom. Then it will determine which atom is the donor one and will determine the position of the other atoms constituting the ligand. 

Useful functions for the placement of coordinates in space were defined: `vec_add()` (add two vectors), `vec_sub()` (substract two vectors) , `vec_scale()` (multiply a vector by a constant), `vec_dot()` (scalar product), `cross()` (give a perpendicular vector to two others), `vec_unit()` (transform a vector in length 1) , norm (give a vector's norm ) , `rotate_vector()`.
Have also been defined functions to place ligands bidentate et tridentate: `translate_bidentate_ligand()` and `translate_tridentate_ligand()`. Indeed, the functions take the bidentate ligand created with RDKit and place them by identifying where should be the donor atoms and what is in between.
For diatomic molecules, `diatomic_ligand_position()` is used to display the diatomic ligand so that they are all well oriented in space. Indeed, at the beginning there were ligands like CN- for which the second atoms (that is not the donor one) were all oriented in the same direction in the complex. Thus, this function allowed to oriented each ligand in the opposite direction of the metal's one. 

The geometry of the complex with the metal at the center was identified above. The donor atoms are known but the geometry of the ligands is not yet. That is why specific functions are defined for each ligand that needs to obey to a certain geometry: `ammonia_ligand_positions()`, `pyridine_ligand_positions()`, `triethyl_ligand_positions()`,  `trimethyl_ligand_positions()`,  `terpyridinel_ligand_positions()`. Without these functions for example the hydrogens in ammonia would all be oriented in the same direction while they should have a triangular planar geometry. Same for pyridine which was not represented in the good plane at the beginning.

Then, `build_complex_3D()` is used to build the complex with the previous functions: first it takes the geometry from `predict_geometry()` and applies `geometry_positions()`. For each ligand it takes its smile, finds the denticity and acts differently if it is monodentate, bidentate or tridentate, using the corresponding translation functions described above.
Then, if the ligand symbol corresponds to a specific ligand needing a characteristic treatment, it applies its respective function defined above.
`view_complex_3D()` is the function used to display the molecule with py3Dmol. It first applies `_to_parsed()` to the input to transform the name or formula of the complex into ParsedComplex format, which can then be exploited and build the molecule calling the `build_complex_3D()` function.
Finally, `complex_3D_html()` transforms the 3D visualisation into an HTLM code used on the app streamlit, which allows to display the complex.

Thus, this part allows to obtain a 3D visualisation of the name or formula of the complex input. However, it allows to display a molecule qualitatively since it is following an idealized geometry. Indeed, the steric effects are not taken into account and the distances used are not proportional, they are just selected to obtain a good visualisation.







### 1.6 Web Application - Streamlit interface

victor

### 1.7 Testing and Validation

The package was validated using a pytest test suite. The tests were organised by module in order to check each part of the workflow separately before testing more integrated behaviours. This was important because the package combines several steps: parsing the input, extracting chemical properties, predicting the geometry, generating visualisations and predicting spectral bands.

The parser was tested in `test_parser.py`. These tests check that formulas such as `[Fe(CN)6]4-`, `[PtCl2(NH3)2]`, `[Co(en)3]3+` and `K4[Fe(CN)6]` are correctly interpreted. They verify the extraction of the metal, ligands, complex charge, counter-ions, ligand metadata, oxidation state and coordination number. Edge cases were also tested, such as unknown ligands, missing charge suffixes, whitespace in the input and invalid metal symbols.

The name parser was tested in `test_name.py`. These tests verify that names such as `tetraamminecopper(II)`, `hexacyanoferrate(II)` and `diamminedichloroplatinum(II)` are converted into the same type of `ParsedComplex` object as formula inputs. They also check the extraction of oxidation states from Roman numerals, the recognition of metal names and the parsing of ligand prefixes such as di-, tetra- and hexa-.

The `Complex` class was tested in `test_complex.py`. These tests check that `Complex.from_formula()`, `Complex.from_name()` and `Complex.from_input()` correctly build a high-level `Complex` object. They also verify that the class exposes the expected properties, including the metal, ligands, oxidation state, coordination number, predicted geometry and d-electron count.

The geometry module was tested in `test_geometry.py`. These tests validate the rule-based geometry prediction for common textbook examples, such as linear silver complexes, octahedral hexacyanoferrate, square-planar platinum(II) complexes and ambiguous four-coordinate nickel or copper complexes. The same test file also checks the calculation of the metal d-electron count.

The 2D visualisation module was tested in `test_diagram_2d.py`. These tests check that SVG diagrams are generated correctly, that files can be saved, and that different geometries produce the expected idealised projections. More specific tests validate the treatment of chelating ligands, EDTA, mixed polydentate/monodentate complexes, compact labels for small ligands, wedge/dash bond styles and square-antiprismatic projections.

The 3D visualisation module was tested in `test_structure_3d.py`. These tests check the generation of idealised coordination-site positions, the creation of 3D ligand structures from SMILES strings, donor atom detection and the construction of a complete RDKit molecule with a 3D conformer. Complexes containing ambidentate ligands were tested, as well as bidentates, tridentates and ligands with a specified geometry such as aromatic ones. The tests also verify that the `Complex` class can access the 3D building and HTML display functions.

Finally, the spectral prediction part was tested using `test_ir_ra_bands.py` and `test_predictor.py`. These tests verify that the spectral database contains the expected ligands and band records, that IR and Raman bands can be extracted, and that the predictor returns a `PredictionResult` object with bands sorted by increasing wavenumber. Benchmark complexes such as hexacyanoferrate, cisplatin and chromium hexacarbonyl were used to check that characteristic spectral regions are represented.

Overall, the tests show that the main workflow of the package is functional: a complex can be parsed from a formula or a name, converted into chemical properties, assigned a predicted geometry, visualised in 2D or 3D, and connected to spectral prediction.

## Results

### 2.1 Representative complexes and extracted properties

The package was first tested on representative coordination complexes chosen to cover the main chemical situations handled by the workflow. The examples include tridentate coordination, mixed bidentate/monodentate ligands, HSAB-based donor assignment, ambiguous four-coordinate geometry, and higher coordination numbers.

The aim of this section is not to show every possible complex, but to demonstrate that the parser and the `Complex` interface can extract chemically relevant information from different types of input. The extracted properties include the metal centre, oxidation state, coordination number, d-electron count, donor atoms, counter-ions when present, and predicted geometry.

<p align="center"><b>Table 1.</b> Representative complexes used to test parsing, property extraction and geometry prediction.</p>

| Input | Feature tested | Oxidation state | CN | d-count | Predicted geometry | Main output |
|---|---|---:|---:|---:|---|---|
| `[Ru(tpy)2]2+` | octahedral complex with two tridentate ligands | +2 | 6 | d6 | octahedral | tridentate coordination |
| `[Cr(en)2(NCS)2]+` | mixed bidentate and monodentate ligands | +3 | 6 | d3 | octahedral | mixed ligand environment |
| `[Pt(dmso)4]2+` | HSAB-based donor assignment for a soft metal centre | +2 | 4 | d8 | square planar | DMSO donor choice |
| `[NiCl4]2-` | ambiguous four-coordinate complex | +2 | 4 | d8 | tetrahedral or square planar | geometry ambiguity |
| `[Fe(CO)5]` | five-coordinate complex | 0 | 5 | d8 | trigonal bipyramidal or square pyramidal | two possible CN5 geometries |
| `K4[Mo(CN)8]` | eight-coordinate complex with counter-ions | +4 | 8 | d2 | square antiprismatic or dodecahedral | counter-ion extraction and CN8 ambiguity |

These examples show that the program can process more than simple monodentate complexes. It correctly accounts for ligand denticity when calculating the coordination number, as shown by the tridentate `[Ru(tpy)2]2+` complex and the mixed-ligand `[Cr(en)2(NCS)2]+` complex. The DMSO example shows that donor-atom assignment can be refined using a chemical rule before the visualisation modules are used.

The examples `[NiCl4]2-`, `[Fe(CO)5]` and `K4[Mo(CN)8]` are especially useful because they show cases where several geometries are chemically possible. Instead of forcing a unique geometry, the program reports the ambiguity. This is appropriate for an educational tool, because the final geometry may depend on ligand field strength, steric effects, experimental conditions or additional structural information.

The `K4[Mo(CN)8]` example also shows that counter-ions written outside the coordination sphere can be extracted. In this case, the four potassium ions are recognised as counter-ions, allowing the program to infer the `4-` charge of the complex ion before calculating the oxidation state of molybdenum.

### 2.2 Two-dimensional visualisation outputs

The 2D visualisation module was tested on the same representative complexes as in Table 1. The objective was to verify that the chemical information extracted by the parser could be converted into readable schematic representations of the coordination sphere.

The SVG format was used because it provides scalable figures that remain clear in the notebook and in exported versions of the report. The examples below illustrate tridentate coordination, mixed bidentate/monodentate ligands, HSAB-based donor assignment, ambiguous four-coordinate geometry, and higher coordination numbers.

<p align="center"><b>Figure 1.</b> 2D visualisation of <code>[Ru(tpy)2]2+</code>, an octahedral complex with two tridentate ligands.</p>

<p align="center">
<img src="../outputs/ru_tpy2_2d.svg" width="450">
</p>

<p align="center"><b>Figure 2.</b> 2D visualisation of <code>[Cr(en)2(NCS)2]+</code>, a mixed bidentate/monodentate complex.</p>

<p align="center">
<img src="../outputs/cr_en2ncs2_2d.svg" width="450">
</p>

<p align="center"><b>Figure 3.</b> 2D visualisation of <code>[Pt(dmso)4]2+</code>, used to test DMSO donor-atom assignment.</p>

<p align="center">
<img src="../outputs/pt_dmso4_2d.svg" width="450">
</p>

<p align="center"><b>Figure 4.</b> 2D visualisation of <code>[NiCl4]2-</code>, an ambiguous four-coordinate complex.</p>

<p align="center">
<img src="../outputs/ni_cl4_2d.svg" width="450">
</p>

<p align="center"><b>Figure 5.</b> 2D visualisation of <code>[Fe(CO)5]</code>, a five-coordinate complex.</p>

<p align="center">
<img src="../outputs/fe_co5_2d.svg" width="450">
</p>

<p align="center"><b>Figure 6.</b> 2D visualisation of <code>K4[Mo(CN)8]</code>, an eight-coordinate complex with counter-ions.</p>

<p align="center">
<img src="../outputs/k4_mo_cn8_2d.svg" width="450">
</p>

The generated drawings are schematic rather than experimentally optimised structures. Their purpose is to represent the connectivity, denticity and relative ligand arrangement around the metal centre in a clear and compact way. This is especially useful for chelating ligands, where the drawing must preserve the relation between the donor atoms and the metal centre.

For higher coordination numbers, such as CN = 5 and CN = 8, the drawings should be interpreted as idealised representations. They are useful for visual comparison, but they do not uniquely determine the experimental geometry when several arrangements are chemically possible.

### 2.3 3D visualisation outputs
Markdown text + screenshots PNG
visualisations issues


### 2.4 Spectral prediction outputs and database coverage 
 IR spectroscopy is widely used to characterize coordination compounds, especially for identifying ligands such as CO, CN⁻, and NO, and distinguishing binding modes (e.g., SCN vs NCS). However, accurate spectral prediction typically requires quantum chemical calculations, which are computationally intensive and not always accessible. There is therefore value in a lightweight, interpretable tool that can provide rapid, approximate spectralpredictionsand assist with spectral assignment. The focus is on interpretability and rapid qualitative analysis rather than high-precision prediction.     
The scope is restricted to a selected set of common ligands : CN, NC, CO, NO, Cl, Br, I, F, OH, O, S, NH3, H2O, NO2, ONO, SCN, NCS, N3, en, phen, bipy, ox, acac, EDTA, Cp, tpy, py, dmso, PPh3, PMe3, PEt3, CH3.  The tool focuses on characteristic group frequencies rather than full vibrational analysis.    
The results is a curated spectra for IR and Raman, taking into account the "terminal" and "chelating" bands associated to the ligands. 
For IR spectra, the plot can have the peaks upwards or downwards, transmittance or absorbance style. 

### 2.5 Streamlit interface output
Markdown text + screenshot


### 2.6 Testing results
Short text + pytest summary

## Discussion


### 3.1 Accuracy and limitations

The current implementation provides a useful educational representation of coordination complexes, but it remains limited to relatively simple systems. The package is mainly designed for mononuclear complexes, meaning complexes containing one metal centre. It does not currently handle polynuclear complexes, metal clusters or complexes with bridging ligands. For example, ligands such as μ-Cl, μ-CN or bridging carbonyls are not explicitly represented as links between two metal centres. This is an important limitation, because many real coordination compounds contain more than one metal centre or bridging coordination modes.

The parser is also based on simplified chemical notation. Formula parsing works well for common coordination formulas, and name parsing works for names containing known ligand names, metal names, prefixes and oxidation states written as Roman numerals. However, the program does not yet support all possible IUPAC naming conventions. For example, it currently uses names such as `tetraamminecopper(II)`, where the Roman numeral gives the metal oxidation state. Some nomenclature systems can also indicate the charge of the coordination sphere directly, for example with numerical charge descriptors. In such cases, the oxidation state would need to be deduced from the total ligand charge and the charge of the complex. This more general name interpretation is not yet implemented.

The visualisation modules are also idealised. In 2D, the program draws only the coordination sphere and does not represent counter-ions around the complex. Small monodentate ligands, such as NH3, H2O, CN or Cl, are often shown using abbreviated labels to keep the drawing readable. This makes the figures clearer, but it also means that the complete molecular structure of these ligands is not always shown. Larger ligands are drawn from SMILES using RDKit, but the representation is still schematic and does not correspond to a crystallographic structure.

The 2D layout is based on idealised geometries and simplified projections. This works well for common geometries such as linear, square planar, tetrahedral and octahedral complexes. However, more complex geometries are harder to represent accurately in a flat drawing. For example, dodecahedral environment can appear flattened in 2D, and not all of them use wedge or dashed bonds because the depth information becomes difficult to represent clearly. Similarly, cyclic ligands are generally drawn flat, although in real complexes they may adopt conformations with more three-dimensional depth.

The 3D visualisation is also limited by the complexity of the ligands. The current module can generate idealised 3D structures for relatively simple ligands, but more complex polydentate ligands are harder to position correctly. For this reason, ligands such as dien or tren were not fully implemented, because their flexible multidentate coordination would require more advanced conformational placement. The current 3D structures should therefore be understood as approximate educational models rather than experimentally optimised geometries.

Finally, the spectral prediction is approximate. The IR and Raman spectra are generated from reference band ranges and simple Gaussian peaks, not from quantum chemical vibrational calculations. This makes the spectra fast and interpretable, but they should not be considered as high-precision predictions. The aim is to help students identify characteristic ligand vibrations and trends, not to replace experimental spectra or computational chemistry.

### 3.2 Value and comparison with existing tools

The main value of CoordAnalyst is that it brings several coordination chemistry tasks together in one educational workflow. Starting from a simple formula or name, the user can obtain the metal and ligands, oxidation state, coordination number, predicted geometry, d-electron count, 2D and 3D visualisations, and approximate IR or Raman spectra. This helps connect concepts that are often taught separately, such as nomenclature, ligand denticity, geometry and vibrational spectroscopy.

Compared with general cheminformatics tools such as RDKit, CoordAnalyst is more specialised for coordination complexes. RDKit is used for molecular representation and drawing, but it does not directly calculate coordination-specific properties such as ligand denticity, metal oxidation state or coordination geometry. CoordAnalyst adds these chemical rules on top of RDKit to make the results more useful for coordination chemistry.

The package is less accurate than molecular visualisation or quantum chemistry software, because it does not use experimental structures, geometry optimisation or ab initio vibrational calculations. However, this is also what makes it fast, transparent and accessible. It can generate idealised structures and approximate spectra directly from user input, without requiring a pre-existing structure file or advanced computational setup.

CoordAnalyst is therefore not intended to replace professional modelling or spectroscopy tools. Its value lies in providing a lightweight, interpretable and modular support tool for teaching and learning coordination chemistry. New ligands, spectral bands or visualisation rules can be added without rewriting the whole package.

### 3.3 Future Improvements

Several improvements could be made to extend the current package. First, the input interpretation could be made more complete. The parser could recognise stereochemical descriptors such as `cis`, `trans`, `fac`, `mer`, `Λ` and `Δ`, store them in the parsed object, and pass them to the 2D and 3D visualisation modules. This would allow the program to generate the requested isomer when the stereochemistry is explicitly specified, while keeping the current idealised representation when it is not.

The input could also be made more flexible by allowing the user to draw the complex directly in the web application. This could be done by integrating a chemical structure editor such as Ketcher or JSME into the Streamlit interface. The program could then extract the metal centre, ligands, donor atoms and connectivity from the drawing and convert this information into the same internal representation used by the formula and name parsers.

Another important improvement would be to extend the range of complexes that can be represented. The current package is mainly designed for mononuclear complexes with terminal ligands. Future versions could include polynuclear complexes, bridging ligands and metal clusters. For example, ligands such as μ-Cl, μ-CN or bridging carbonyls would need to be represented as connections between several metal centres. This would require a more general internal representation of the complex and new drawing rules for both 2D and 3D visualisation.

The geometry prediction could also be made more transparent by adding confidence levels such as `probable`, `ambiguous` or `user-selected`. The interface could display the reasoning behind the proposed geometry, including the coordination number, metal, oxidation state, ligand type and d-electron count. This would be especially useful when several geometries are chemically possible.

The visualisation modules could then be improved using this richer chemical information. In 2D, more realistic depth information could be added for cyclic ligands and complex geometries such as dodecahedral or square-antiprismatic coordination. In 3D, more advanced ligand-placement algorithms could be used for flexible polydentate ligands such as dien or tren. Approximate metal--ligand bond distances could also be included using experimental data or a curated database, while keeping idealised drawings as the default for readability.

The spectral prediction could also be developed further. In the current implementation, simplified selection rules are applied only for highly symmetric cases: octahedral complexes with six identical ligands use an `Oh`-type model, tetrahedral complexes with four identical ligands use a `Td`-type model, and square-planar complexes with four identical ligands use a `D4h`-type model. For mixed-ligand complexes or less standard geometries, the program mainly identifies the relevant metal--ligand and ligand-internal stretching or bending modes and assumes that these modes are active.

A future version could estimate approximate point groups for a wider range of complexes and use them to apply more realistic IR and Raman selection rules. IR activity could be linked to the transformation properties of the dipole-moment components, while Raman activity could be linked to the transformation properties of the polarizability components. This would make the predicted spectra depend more directly on the symmetry, geometry and ligand arrangement of the complex.

The spectral database could also be expanded so that band positions depend more specifically on the metal, oxidation state, geometry and surrounding ligands. This would improve the prediction of metal--ligand vibrations and of ligands whose stretching frequencies are strongly affected by backbonding, such as CO, CN and NO.

Finally, the program could include possible distortions, such as Jahn--Teller elongation, tetragonal distortion or trigonal distortion. These distortions would not necessarily need to modify the 2D or 3D drawings, but they could improve the spectral interpretation, because lowering the symmetry of a complex can split vibrational bands and change which modes are IR or Raman active. The Streamlit interface could also include an “Assumptions” section explaining these simplifications, together with export options such as PNG or PDF figures and comparison with uploaded experimental spectra.